### Import Libraries

In [ ]:
import os

code_path = "../"
exp_path = "./experiments"

os.chdir(code_path)
os.getcwd()

In [ ]:
# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

import cv2
import torch
import mlflow
import logging
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime
from pathlib import Path
from tqdm import tqdm
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch import optim
from src.org_unet_parts import *
from dataloader import ACDCDataset
from utils.dice_score import dice_loss, dice_coeff
from utils.evaluate import evaluate
%matplotlib inline

In [ ]:
os.chdir(exp_path)
os.getcwd()

In [ ]:
exp_run_id = f'acdc_residual_unet_v1_img_slices_with_ratios_v4{datetime.now().strftime("%Y%m%d%H%M%S")}'
log_file_name = f'./logs/{exp_run_id}.log'
logging.basicConfig(filename=log_file_name, level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s', force=True)

In [ ]:
torch.cuda.empty_cache()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logging.info(f'Using device {device}')
print(f'Using device {device}')

In [ ]:
mlflow.login() # host: https://community.cloud.databricks.com/
mlflow.set_tracking_uri("databricks")

# Initialize logging - create a new MLflow Experiment
mlflow.set_experiment("/cmri-segmentation-of-ventricular-structures-and-myocardium")

In [ ]:
dir_checkpoint = Path('./models/checkpoints/residual_unet_v1_img_slices_with_ratios_v4/')

epochs = 10
batch_size = 8
lr = 1e-3
scale = 1
amp = False

### Data Loading

In [ ]:
root_dir = r'../data/ACDC/img_slices_with_ratios_v4/'
logging.info(f'Using root_dir {root_dir}')

training_dataset = ACDCDataset(root_dir=root_dir, dataset='training')
validation_dataset = ACDCDataset(root_dir=root_dir, dataset='validation')
testing_dataset = ACDCDataset(root_dir=root_dir, dataset='testing')

logging.info(f'Training dataset size: {len(training_dataset)}')
logging.info(f'Validation dataset size: {len(validation_dataset)}')
logging.info(f'Testing dataset size: {len(testing_dataset)}')

# Load data from the dataset
train_dataloader = DataLoader(training_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(testing_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
# Iterate through the dataset and plot the first 4 samples
n_samples_to_plot = 2

for i, sample in enumerate(test_dataloader):
    if i >= n_samples_to_plot:
        break
    
    # Extract data from the sample
    image = sample['image'][i].squeeze().numpy()  # Convert to numpy array
    msk_all = sample['masks_all'][i].squeeze().numpy()
    msk_bkg = sample['masks'][i][0].squeeze().numpy()
    msk_lv = sample['masks'][i][1].squeeze().numpy()
    msk_rv = sample['masks'][i][2].squeeze().numpy()
    msk_myo = sample['masks'][i][3].squeeze().numpy()
    
    # Create a figure with subplots
    fig, axes = plt.subplots(1, 6, figsize=(12, 5))
    
    # Plot the image and masks
    axes[0].imshow(image, cmap='gray')
    axes[0].set_title('Image')
    axes[0].axis('off')
    
    axes[1].imshow(msk_bkg, cmap='gray')
    axes[1].set_title('Mask Background')
    axes[1].axis('off')
    
    axes[2].imshow(msk_lv, cmap='gray')
    axes[2].set_title('Mask LV')
    axes[2].axis('off')
    
    axes[3].imshow(msk_rv, cmap='gray')
    axes[3].set_title('Mask RV')
    axes[3].axis('off')
    
    axes[4].imshow(msk_myo, cmap='gray')
    axes[4].set_title('Mask Myo')
    axes[4].axis('off')
    
    axes[5].imshow(msk_all, cmap='gray')
    axes[5].set_title('Combined Mask')
    axes[5].axis('off')
    
    # Adjust layout and show the plot
    plt.tight_layout()
    plt.show()

### Residual U-Net Model

In [ ]:
# Define a residual block
class ResidualConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ResidualConvBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU(inplace=True)
        self.batchnorm1 = nn.BatchNorm2d(out_channels)
        self.batchnorm2 = nn.BatchNorm2d(out_channels)

        # Residual connection (1x1 convolution to match dimensions if in_channels != out_channels)
        self.residual = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else None

    def forward(self, x):
        residual = x  # Store the input for the residual connection
        x = self.relu(self.batchnorm1(self.conv1(x)))
        x = self.batchnorm2(self.conv2(x))

        # Apply residual connection
        if self.residual:
            residual = self.residual(residual)

        x += residual  # Add the residual connection
        x = self.relu(x)  # Apply ReLU after adding residual
        return x

# Define the Residual UNet
class ResidualUNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=4):
        super(ResidualUNet, self).__init__()
        self.n_channels = in_channels
        self.n_classes = out_channels
        self.bilinear = False

        # Encoder
        self.encoder1 = ResidualConvBlock(in_channels, 64)
        self.encoder2 = ResidualConvBlock(64, 128)
        self.encoder3 = ResidualConvBlock(128, 256)
        self.encoder4 = ResidualConvBlock(256, 512)

        self.pool = nn.MaxPool2d(2, 2)

        # Bottleneck
        self.bottleneck = ResidualConvBlock(512, 1024)

        # Decoder
        self.upconv4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.decoder4 = ResidualConvBlock(1024, 512)

        self.upconv3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.decoder3 = ResidualConvBlock(512, 256)

        self.upconv2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.decoder2 = ResidualConvBlock(256, 128)

        self.upconv1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.decoder1 = ResidualConvBlock(128, 64)

        # Output layer
        self.output = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder
        e1 = self.encoder1(x)
        e2 = self.encoder2(self.pool(e1))
        e3 = self.encoder3(self.pool(e2))
        e4 = self.encoder4(self.pool(e3))

        # Bottleneck
        b = self.bottleneck(self.pool(e4))

        # Decoder with residual connections
        d4 = self.upconv4(b)
        d4 = torch.cat((d4, e4), dim=1)  # Concatenate skip connection
        d4 = self.decoder4(d4)

        d3 = self.upconv3(d4)
        d3 = torch.cat((d3, e3), dim=1)  # Concatenate skip connection
        d3 = self.decoder3(d3)

        d2 = self.upconv2(d3)
        d2 = torch.cat((d2, e2), dim=1)  # Concatenate skip connection
        d2 = self.decoder2(d2)

        d1 = self.upconv1(d2)
        d1 = torch.cat((d1, e1), dim=1)  # Concatenate skip connection
        d1 = self.decoder1(d1)

        # Output layer
        output = self.output(d1)
        return output

### Model Training

In [ ]:
def train_model(
        model,
        device,
        epochs: int = 5,
        batch_size: int = 1,
        learning_rate: float = 1e-5,
        save_checkpoint: bool = True,
        img_scale: float = 0.5,
        amp: bool = False,
        weight_decay: float = 1e-8,
        momentum: float = 0.999,
        gradient_clipping: float = 1.0,
):
    n_train = len(training_dataset)
    n_val = len(validation_dataset)
    
    # Start an MLflow run
    with mlflow.start_run() as run:
        # Get the run_id of the current active run
        run_id = run.info.run_id

        logging.info(f'''Starting training:
            Epochs:          {epochs}
            Batch size:      {batch_size}
            Learning rate:   {learning_rate}
            Training size:   {n_train}
            Validation size: {n_val}
            Checkpoints:     {save_checkpoint}
            Device:          {device.type}
            Images scaling:  {img_scale}
            Mixed Precision: {amp}
        ''')
        
        params = {
            'epochs': epochs,
            'batch_size': batch_size,
            'learning_rate': learning_rate,
            'training_set_size': n_train,
            'validation_set_size': n_val,
            'img_scale': img_scale,
            'mixed_precision': amp
        }
        
        mlflow.log_params(params)
        
        # Initialize TensorBoard writer
        writer = SummaryWriter(log_dir=f'./logs/tensorboard_runs/{exp_run_id}')  # This will save logs in '/logs/tensorboard_runs/{exp_run_id}'

        # Set up the optimizer, the loss, the learning rate scheduler and the loss scaling for AMP
        # optimizer = optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay, momentum=momentum, foreach=True)
        optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay, foreach=True)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'max', patience=5)  # goal: maximize Dice score
        grad_scaler = torch.cuda.amp.GradScaler(enabled=amp)
        # criterion = nn.BCELoss() if model.n_classes > 1 else nn.BCEWithLogitsLoss() # nn.CrossEntropyLoss()
        criterion = nn.BCEWithLogitsLoss()
        global_step = 0

        # Begin training
        for epoch in range(1, epochs + 1):
            model.train()
            epoch_loss = 0

            with tqdm(total=n_train, desc=f'Epoch {epoch}/{epochs}', unit='img') as pbar:
                for batch in train_dataloader:
                    # images, msks_lv, msks_rv, msks_myo = batch['image'], batch['mask_lv'], batch['mask_rv'], batch['mask_myo']
                    images, masks, masks_all = batch['image'], batch['masks'], batch['masks_all']
                    images = images.permute(0, 3, 1, 2)
                    # masks_all = torch.argmax(masks_all, dim=1)
                    masks_all = masks_all.permute(0, 3, 1, 2)
                    # logging.info(f"images shape: {images.shape}")
                    # logging.info(f"true masks shape: {masks.shape}")
    
                    assert images.shape[1] == model.n_channels, \
                        f'Network has been defined with {model.n_channels} input channels, ' \
                        f'but loaded images have {images.shape[1]} channels. Please check that ' \
                        'the images are loaded correctly.'
    
                    images = images.to(device=device, dtype=torch.float32, memory_format=torch.channels_last)
                    true_masks = masks.to(device=device, dtype=torch.float32, memory_format=torch.channels_last)
                    # true_masks = masks_all.to(device=device, dtype=torch.float32, memory_format=torch.channels_last)
    
                    with torch.autocast(device.type if device.type != 'mps' else 'cpu', enabled=amp):
                        # masks_pred = model(images).permute(1, 0, 2, 3)
                        masks_pred = model(images)
                        # logging.info(f"masks pred shape: {masks_pred.shape}")
                        if model.n_classes == 1:
                            loss_ce = criterion(masks_pred.squeeze(1), true_masks.float())
                            loss_dice = dice_loss(F.sigmoid(masks_pred.squeeze(1)), true_masks.float(), multiclass=False)
                            loss = 0.4 * loss_ce + 0.6 * loss_dice
                        else:
                            loss_ce = criterion(masks_pred, true_masks)  # true_masks.squeeze(1).long()
                            loss_dice = dice_loss(
                                # masks_pred,
                                # true_masks,
                                F.softmax(masks_pred, dim=1).float(),
                                # F.sigmoid(masks_pred > 0.5).float(),
                                true_masks.float(),
                                # F.softmax(masks_pred, dim=1).float(),
                                # F.one_hot(true_masks, model.n_classes).permute(0, 3, 1, 2).float(),
                                multiclass=True
                            )
                            loss = 0.4 * loss_ce + 0.6 * loss_dice
    
                    optimizer.zero_grad(set_to_none=True)
                    grad_scaler.scale(loss).backward()
                    grad_scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clipping)
                    grad_scaler.step(optimizer)
                    grad_scaler.update()
    
                    pbar.update(images.shape[0])
                    global_step += 1
                    epoch_loss += loss.item()
                    pbar.set_postfix(**{'loss (batch)': loss.item()})
    
                    # Evaluation round
                    division_step = (n_train // (5 * batch_size))
                    if division_step > 0:
                        if global_step % division_step == 0:
                            val_score = evaluate(model, val_dataloader, device, amp)
                            scheduler.step(val_score)
                            logging.info('Validation Dice score: {}'.format(val_score))
    
            if save_checkpoint:
                Path(dir_checkpoint).mkdir(parents=True, exist_ok=True)
                state_dict = model.state_dict()
                # state_dict['mask_values'] = training_dataset.mask_values
                torch.save(state_dict, str(dir_checkpoint / 'checkpoint_epoch{}.pth'.format(epoch)))
                logging.info(f'Checkpoint {epoch} saved!')

            # Log metrics to TensorBoard
            avg_loss = epoch_loss / len(train_dataloader)
            writer.add_scalar("Loss", avg_loss, epoch)
        
        # Close the writer when done
        writer.close()
        
        '''
        # Log train, validation and test set Dice Scores
        train_dice_score = evaluate(model, train_dataloader, device, amp).item()
        val_dice_score = evaluate(model, val_dataloader, device, amp).item()
        test_dice_score = evaluate(model, test_dataloader, device, amp).item()
        
        mlflow.log_metric("train_dice_score", round(train_dice_score, 4))
        mlflow.log_metric("val_dice_score", round(val_dice_score, 4))
        mlflow.log_metric("test_dice_score", round(test_dice_score, 4))
        
        logging.info(f'Overall Train Set Dice score: {round(train_dice_score, 4)}')
        logging.info(f'Overall Validation Set Dice score: {round(val_dice_score, 4)}')
        logging.info(f'Overall Test Dice Set score: {round(test_dice_score, 4)}')
        '''
    
    return run_id

### Training

In [ ]:
# n_channels=3 for RGB images
# n_classes is the number of probabilities you want to get per pixel
model = ResidualUNet(in_channels=1, out_channels=4)
model = model.to(memory_format=torch.channels_last)

logging.info(f'Network:\n'
                 f'\t{model.n_channels} input channels\n'
                 f'\t{model.n_classes} output channels (classes)\n'
                 f'\t{"Bilinear" if model.bilinear else "Transposed conv"} upscaling')

model.to(device=device)

try:
    run_id = train_model(
        model=model,
        epochs=epochs,
        batch_size=batch_size,
        learning_rate=lr,
        device=device,
        img_scale=scale,
        amp=amp
    )
except torch.cuda.OutOfMemoryError:
    logging.error('Detected OutOfMemoryError! '
                  'Enabling checkpointing to reduce memory usage, but this slows down training. '
                  'Consider enabling AMP (--amp) for fast and memory efficient training')
    torch.cuda.empty_cache()
    model.use_checkpointing()
    run_id = train_model(
        model=model,
        epochs=epochs,
        batch_size=batch_size,
        learning_rate=lr,
        device=device,
        img_scale=scale,
        amp=amp
    )

### Visualize Model Outputs

In [ ]:
def visualize_prediction(model, dataloader, device, sample_idx=0, img_idx=0, n_classes=4, combined_mask=True):
    plt.style.use("classic")
    model.eval()
    with torch.no_grad():
        for i, sample in enumerate(dataloader):
            if i == sample_idx:
                # Extract data from the sample
                images = sample['image']
                true_masks = sample['masks']
                true_masks_all = sample['masks_all']
                print(f'images shape: {images.shape}')
                print(f'true masks shape: {true_masks.shape}')
                print(f'combined true masks shape: {true_masks_all.shape}')
                images = images.permute(0, 3, 1, 2)
                true_masks_all = true_masks_all.permute(0, 3, 1, 2)
                
                images = images.to(device=device, dtype=torch.float32, memory_format=torch.channels_last)
                # true_masks = true_masks.to(device=device, dtype=torch.float32, memory_format=torch.channels_last)
                # true_masks_all = true_masks_all.to(device=device, dtype=torch.float32, memory_format=torch.channels_last)
                # true_masks = (true_masks * 255).round().long()
                true_masks_all = (true_masks_all * 255).round().long()

                outputs = model(images)

                # Apply sigmoid to convert logits to probabilities, then threshold
                # preds = torch.sigmoid(outputs)
                preds = (outputs > 0.5).float()  # Convert to binary masks (0 or 1)
                # preds = preds.float().cpu().numpy().squeeze(1)
                
                # Convert logits to predicted class labels
                # preds = torch.argmax(outputs, dim=1).long()
                
                image = images[img_idx].cpu().squeeze()
                true_mask_all = true_masks_all[img_idx].cpu().squeeze().numpy()
                true_mask_bkg = true_masks[img_idx][0].cpu().squeeze().numpy()
                true_mask_lv = true_masks[img_idx][1].cpu().squeeze().numpy()
                true_mask_rv = true_masks[img_idx][2].cpu().squeeze().numpy()
                true_mask_myo = true_masks[img_idx][3].cpu().squeeze().numpy()
                
                # Apply argmax to get the class with the highest probability for each pixel
                pred_mask_all = None
                # pred_mask_all = torch.argmax(preds[img_idx], dim=0).cpu().numpy()  # Shape: (height, width)
                # pred_mask_all = preds[img_idx].cpu().numpy()
                
                print(f'preds shape: {preds.shape}')
                print(f'True mask unique values: {np.unique(true_mask_all)}')
                # print(f'Pred mask unique values: {np.unique(pred_mask_all)}')
                
                if n_classes > 1 and combined_mask:
                    fig, ax = plt.subplots(1, 6, figsize=(16, 5), facecolor='white')
                    ax[0].imshow(image, cmap='gray')
                    ax[0].set_title('Input Image')
    
                    ax[1].imshow(true_mask_lv, cmap='gray')
                    ax[1].set_title('True LV')
    
                    ax[2].imshow(true_mask_rv, cmap='gray')
                    ax[2].set_title('True RV')
    
                    ax[3].imshow(true_mask_myo, cmap='gray')
                    ax[3].set_title('True MYO')
                    
                    ax[4].imshow(true_mask_all, cmap='viridis')
                    ax[4].set_title('True Mask')
                    
                    ax[5].imshow(pred_mask_all, cmap='viridis')
                    ax[5].set_title('Predicted Mask')
                elif n_classes > 1 and not combined_mask:
                    fig, ax = plt.subplots(2, 5, figsize=(20, 8), facecolor='white')
                    ax[0][0].imshow(image, cmap='gray')
                    ax[0][0].set_title('Input Image')
                    
                    ax[0][1].imshow(true_mask_bkg, cmap='gray')
                    ax[0][1].set_title('True Background')
    
                    ax[0][2].imshow(true_mask_lv, cmap='gray')
                    ax[0][2].set_title('True LV')
    
                    ax[0][3].imshow(true_mask_rv, cmap='gray')
                    ax[0][3].set_title('True RV')
    
                    ax[0][4].imshow(true_mask_myo, cmap='gray')
                    ax[0][4].set_title('True MYO')
    
                    ax[1][0].imshow(image, cmap='gray')
                    ax[1][0].set_title('Input Image')
                    
                    ax[1][1].imshow(preds[img_idx][0].cpu().squeeze().numpy(), cmap='gray')
                    ax[1][1].set_title('Pred Background')
    
                    ax[1][2].imshow(preds[img_idx][1].cpu().squeeze().numpy(), cmap='gray')
                    ax[1][2].set_title('Pred LV')
    
                    ax[1][3].imshow(preds[img_idx][2].cpu().squeeze().numpy(), cmap='gray')
                    ax[1][3].set_title('Pred RV')
    
                    ax[1][4].imshow(preds[img_idx][3].cpu().squeeze().numpy(), cmap='gray')
                    ax[1][4].set_title('Pred MYO')
                else:
                    print(f'Image: {torch.unique(images[img_idx])}')
                    print(f'True Mask: {torch.unique(true_masks_all[img_idx])}')
                    print(f'Pred Mask: {np.unique(preds[img_idx])}')
                    
                    fig, ax = plt.subplots(2, 2, figsize=(8, 8), facecolor='white')
                    ax[0][0].imshow(image, cmap='gray')
                    ax[0][0].set_title('Input Image')
                    
                    ax[0][1].imshow(true_mask_all, cmap='gray')
                    ax[0][1].set_title('True Mask')
                    
                    ax[1][0].imshow(image, cmap='gray')
                    ax[1][0].set_title('Input Image')
                    
                    ax[1][1].imshow(pred_mask_all, cmap='gray')
                    ax[1][1].set_title('Predicted Mask')

                plt.show()
                break  # Visualize only one image
            else:
                continue

In [ ]:
visualize_prediction(model, test_dataloader, device, sample_idx=1, img_idx = 6, n_classes=4, combined_mask=False)

### Model Evaluation

In [ ]:
# Function to calculate Dice score for a single class
def dice_score(pred, target, smooth=1e-6):
    intersection = (pred * target).sum()
    sets_sum = pred.sum() + target.sum()
    sets_sum = torch.where(sets_sum == 0, intersection, sets_sum)
    dice = (2. * intersection + smooth) / (sets_sum + smooth)
    return dice


# Function to evaluate Dice score for each class in the test set
def evaluate_dice_score(model, dataloader, num_classes=4, device='cpu', threshold=0.5):
    model.eval()  # Set model to evaluation mode
    dice_scores = [0.0] * num_classes  # List to store Dice score for each class
    # count = 0  # Number of samples
    batch_count = len(dataloader) # Number of batches

    with torch.no_grad():
        for i, sample in enumerate(dataloader):
            # Extract data from the sample
            images = sample['image']
            true_masks = sample['masks']
            true_masks_all = sample['masks_all']
            # print(f'images shape: {images.shape}')
            # print(f'true masks shape: {true_masks.shape}')
            # print(f'combined true masks shape: {true_masks_all.shape}')
            images = images.permute(0, 3, 1, 2)
            true_masks_all = true_masks_all.permute(0, 3, 1, 2)

            images = images.to(device=device, dtype=torch.float32, memory_format=torch.channels_last)
            true_masks = true_masks.to(device=device, dtype=torch.float32, memory_format=torch.channels_last)
            
            outputs = model(images)  # Shape: (batch_size, 4, height, width)
            # outputs = torch.sigmoid(outputs)  # Apply sigmoid to get probabilities

            # Binarize outputs based on threshold
            # predictions = (outputs > threshold).float()  # Shape: (batch_size, 4, height, width)
            predictions = (F.sigmoid(outputs) > threshold).float()

            # Calculate Dice score for each class
            for i in range(num_classes):  # Loop over each class
                dice_scores[i] += dice_coeff(
                    predictions[:, i, :, :].flatten(0, 1),
                    true_masks[:, i, :, :].flatten(0, 1),
                    reduce_batch_first=False
                    )
                '''
                dice_scores[i] += dice_score(
                    predictions[:, i, :, :],
                    true_masks[:, i, :, :]
                    )
                '''
            
            # count += images.size(0)  # Update sample count
            # batch_count += 1 # Update batch count
    
    # Average Dice scores across the dataset
    # avg_dice_scores = [score / count for score in dice_scores]
    avg_dice_scores_batch = [score / batch_count for score in dice_scores]
    return avg_dice_scores_batch

In [ ]:
with mlflow.start_run(run_id=run_id):
    # Compute the Dice score for each class in the Train set
    avg_dice_scores_batch = evaluate_dice_score(model, train_dataloader, device=device)
    print("Dice Scores - Train Set:")
    print(f"BKG: {avg_dice_scores_batch[0]:.4f}\tLV: {avg_dice_scores_batch[1]:.4f}\tRV: {avg_dice_scores_batch[2]:.4f}\tMYO: {avg_dice_scores_batch[3]:.4f}\tOVR: {round(torch.tensor(avg_dice_scores_batch[1:]).mean().item(), 4)}")
    mlflow.log_metric("train_dc_bkg", round(avg_dice_scores_batch[0].item(), 4))
    mlflow.log_metric("train_dc_lv", round(avg_dice_scores_batch[1].item(), 4))
    mlflow.log_metric("train_dc_rv", round(avg_dice_scores_batch[2].item(), 4))
    mlflow.log_metric("train_dc_myo", round(avg_dice_scores_batch[3].item(), 4))
    mlflow.log_metric("train_dice_score", round(torch.tensor(avg_dice_scores_batch[1:]).mean().item(), 4))


    # Compute the Dice score for each class in the Validation set
    avg_dice_scores_batch = evaluate_dice_score(model, val_dataloader, device=device)
    print("\nDice Scores - Validation Set:")
    print(f"BKG: {avg_dice_scores_batch[0]:.4f}\tLV: {avg_dice_scores_batch[1]:.4f}\tRV: {avg_dice_scores_batch[2]:.4f}\tMYO: {avg_dice_scores_batch[3]:.4f}\tOVR: {round(torch.tensor(avg_dice_scores_batch[1:]).mean().item(), 4)}")
    mlflow.log_metric("val_dc_bkg", round(avg_dice_scores_batch[0].item(), 4))
    mlflow.log_metric("val_dc_lv", round(avg_dice_scores_batch[1].item(), 4))
    mlflow.log_metric("val_dc_rv", round(avg_dice_scores_batch[2].item(), 4))
    mlflow.log_metric("val_dc_myo", round(avg_dice_scores_batch[3].item(), 4))
    mlflow.log_metric("val_dice_score", round(torch.tensor(avg_dice_scores_batch[1:]).mean().item(), 4))


    # Compute the Dice score for each class in the Test set
    avg_dice_scores_batch = evaluate_dice_score(model, test_dataloader, device=device)
    print("\nDice Scores - Test Set:")
    print(f"BKG: {avg_dice_scores_batch[0]:.4f}\tLV: {avg_dice_scores_batch[1]:.4f}\tRV: {avg_dice_scores_batch[2]:.4f}\tMYO: {avg_dice_scores_batch[3]:.4f}\tOVR: {round(torch.tensor(avg_dice_scores_batch[1:]).mean().item(), 4)}")
    mlflow.log_metric("test_dc_bkg", round(avg_dice_scores_batch[0].item(), 4))
    mlflow.log_metric("test_dc_lv", round(avg_dice_scores_batch[1].item(), 4))
    mlflow.log_metric("test_dc_rv", round(avg_dice_scores_batch[2].item(), 4))
    mlflow.log_metric("test_dc_myo", round(avg_dice_scores_batch[3].item(), 4))
    mlflow.log_metric("test_dice_score", round(torch.tensor(avg_dice_scores_batch[1:]).mean().item(), 4))